# Autograd

PyTorch **Autograd** is the automatic differentiation engine used to compute gradients.

This notebook builds the mental model behind Autograd through small experiments before using it in a training loop.

> **Learning principle:** Understand the mechanism first, then let PyTorch automate it.

## 1. `requires_grad`

A tensor created with `requires_grad=True` tells PyTorch to track operations involving that tensor so gradients can later be computed.

In [50]:
import torch

x = torch.tensor(2.0, requires_grad=True)

print(x)
print(x.requires_grad)
print(x.grad)
print(x.grad_fn)

tensor(2., requires_grad=True)
True
None
None


## 2. Computational Graphs

When tensors requiring gradients participate in operations, PyTorch builds a computational graph.

Example:

```text
x
├── x²
└── 5x
     ↓
     z
```

For

$$
z = x^2 + 5x
$$

the backward pass can use the graph and the chain rule to calculate `dz/dx`.

In [51]:
x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = x * 5
z = a + b

print("z =", z)
print("z.grad_fn =", z.grad_fn)

z = tensor(14., grad_fn=<AddBackward0>)
z.grad_fn = <AddBackward0 object at 0x758a414c8790>


## 3. `grad_fn`

`grad_fn` describes the backward operation associated with a tensor produced by an operation.

It is **not the gradient itself**.

For example, an addition creates an `AddBackward0` node, while a power operation can create a `PowBackward0` node.

In [52]:
print(z.grad_fn)
print(z.grad_fn.next_functions)

((<PowBackward0 object at 0x758a394dc160>, 0), (<MulBackward0 object at 0x758a394dcdf0>, 0))


## 4. Following the Backward Graph

`next_functions` lets us inspect the backward graph.

For:

```text
x → multiplication → addition → z
```

the backward graph is traversed in the opposite direction:

```text
z → AddBackward → MulBackward → x
```

In [53]:
print(z.grad_fn.next_functions)
print(z.grad_fn.next_functions[0][0].next_functions)

((<PowBackward0 object at 0x758a394dc160>, 0), (<MulBackward0 object at 0x758a394dcdf0>, 0))
((<AccumulateGrad object at 0x758a394dc7c0>, 0),)


## 5. Branching Computational Graphs

A tensor can contribute through multiple branches.

For:

$$
z=x^2+5x
$$

the derivative is:

$$
\frac{dz}{dx}=2x+5
$$

At `x = 2`:

$$
\frac{dz}{dx}=9
$$

The gradient contributions from both branches are added during backpropagation.

In [54]:
x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = x * 5
z = a + b

z.backward()

print(x.grad)

tensor(9.)


## 6. `backward()`

Calling:

```python
z.backward()
```

starts the backward pass through the computational graph.

For the previous example, Autograd calculates:

```text
dz/dx = 9
```

and stores it in:

```python
x.grad
```

## 7. A Deeper Computational Graph

Consider:

$$
c=x^2+5x
$$

$$
d=c^2
$$

At `x = 2`:

```text
x = 2
a = x² = 4
b = 5x = 10
c = 14
d = 196
```

The chain rule gives:

$$
\frac{dd}{dc}=28
$$

$$
\frac{dc}{dx}=9
$$

$$
\frac{dd}{dx}=28\times9=252
$$

In [55]:
x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = x * 5
c = a + b
d = c ** 2

d.backward()

print("x.grad =", x.grad)

x.grad = tensor(252.)


## 8. Leaf and Non-Leaf Tensors

`x` was created directly with `requires_grad=True`, so it is a **leaf tensor**.

`a`, `b`, `c`, and `d` were produced by operations, so they are **non-leaf tensors**.

By default, PyTorch stores gradients in `.grad` for leaf tensors that require gradients. Intermediate gradients are not retained unless requested.

In [56]:
print("x.grad =", x.grad)
print("a.grad =", a.grad)
print("b.grad =", b.grad)
print("c.grad =", c.grad)
print("d.grad =", d.grad)

x.grad = tensor(252.)
a.grad = None
b.grad = None
c.grad = None
d.grad = None


/tmp/ipykernel_7259/1665850514.py:2: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /__w/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:493.)
  print("a.grad =", a.grad)
/tmp/ipykernel_7259/1665850514.py:3: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.c

## 9. `retain_grad()`

If we want the gradient of a non-leaf tensor to remain available after `backward()`, we can call:

```python
tensor.retain_grad()
```

In [57]:
x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = x * 5
c = a + b
d = c ** 2

a.retain_grad()
b.retain_grad()
c.retain_grad()
d.retain_grad()

d.backward()

print("x.grad =", x.grad)
print("a.grad =", a.grad)
print("b.grad =", b.grad)
print("c.grad =", c.grad)
print("d.grad =", d.grad)

x.grad = tensor(252.)
a.grad = tensor(28.)
b.grad = tensor(28.)
c.grad = tensor(28.)
d.grad = tensor(1.)


The gradients are:

```text
d.grad = 1
c.grad = 28
a.grad = 28
b.grad = 28
x.grad = 252
```

The two branches through `a` and `b` contribute to the gradient of `x` and those contributions are summed.

## 10. Gradient Accumulation

PyTorch **accumulates gradients by default**.

Calling `backward()` again adds the new gradient to the existing `.grad` value.

In [58]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

print("After first backward:", x.grad)

y = x ** 2
y.backward()

print("After second backward:", x.grad)

After first backward: tensor(4.)
After second backward: tensor(8.)


Because the gradient is accumulated, training loops must clear gradients before the next iteration.

A tensor's gradient can be cleared with:

```python
x.grad.zero_()
```

## 11. `torch.no_grad()`

`torch.no_grad()` temporarily disables gradient tracking for operations inside its block.

This avoids building an Autograd graph and avoids saving information needed for backward computation.

In [59]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2

print(y.requires_grad)
print(y.grad_fn)

False
None


This is especially useful during inference, when gradients are not needed.

## 12. `requires_grad_(False)`

`torch.no_grad()` is a temporary context.

By contrast:

```python
x.requires_grad_(False)
```

changes the tensor's gradient-tracking state.

In [60]:
x = torch.tensor(2.0, requires_grad=True)

print("Before:", x.requires_grad)

x.requires_grad_(False)

print("After:", x.requires_grad)

Before: True
After: False


## 13. `detach()`

`detach()` creates a new tensor that:

- shares the same underlying storage
- is disconnected from the Autograd graph

It does **not** copy the data.

In [61]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
z = y.detach()

print(y.requires_grad)
print(z.requires_grad)
print(y.grad_fn)
print(z.grad_fn)

True
False
None


### Storage sharing

Because `detach()` shares storage, modifying the detached tensor can modify the original tensor's data.

In [62]:
x = torch.tensor([1., 2., 3.], requires_grad=True)

y = x.detach()

y[0] = 100

print("x:", x)
print("y:", y)

x: tensor([100.,   2.,   3.], requires_grad=True)
y: tensor([100.,   2.,   3.])


## 14. `clone()` vs `detach()`

The important distinction:

| Operation | New tensor object | New storage | Autograd connection |
|---|---:|---:|---|
| `y = x` | No | No | Same |
| `y = x.clone()` | Yes | Yes | Preserved when applicable |
| `y = x.detach()` | Yes | No | Disconnected |
| `y = x.clone().detach()` | Yes | Yes | Disconnected |

`clone().detach()` is therefore useful when an independent copy is wanted without gradient tracking.

In [63]:
x = torch.tensor([1., 2., 3.], requires_grad=True)

a = x.detach()
b = x.clone().detach()

a[0] = 100

print("x:", x)
print("b:", b)

x: tensor([100.,   2.,   3.], requires_grad=True)
b: tensor([1., 2., 3.])


# Training with Autograd

Now we connect Autograd to the training process.

We start with a single trainable weight.

## 15. One Trainable Weight

Model:

$$
\hat y = wx
$$

Loss:

$$
L=(\hat y-y)^2
$$

For `x = 3`, `w = 2`, and target `10`:

```text
prediction = 6
loss = 16
dL/dw = -24
```

In [64]:
x = torch.tensor(3.0)
w = torch.tensor(2.0, requires_grad=True)

target = torch.tensor(10.0)

prediction = x * w
loss = (prediction - target) ** 2

loss.backward()

print("Prediction:", prediction)
print("Loss:", loss)
print("w.grad:", w.grad)

Prediction: tensor(6., grad_fn=<MulBackward0>)
Loss: tensor(16., grad_fn=<PowBackward0>)
w.grad: tensor(-24.)


## 16. Gradient Descent

Gradient descent updates parameters using:

$$
w_{new}=w_{old}-\eta\frac{dL}{dw}
$$

With:

```text
w = 2
dL/dw = -24
learning rate = 0.1
```

the update is:

```text
w = 4.4
```

In [65]:
w = torch.tensor(2.0, requires_grad=True)

gradient = torch.tensor(-24.0)
learning_rate = 0.1

with torch.no_grad():
    w -= learning_rate * gradient

print(w)

tensor(4.4000, requires_grad=True)


## 17. Weight and Bias

A neuron can be written as:

$$
\hat y=wx+b
$$

Autograd can calculate gradients for both trainable parameters.

In [66]:
x = torch.tensor(3.0)
target = torch.tensor(10.0)

w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

prediction = w * x + b
loss = (prediction - target) ** 2

loss.backward()

print("Prediction:", prediction)
print("Loss:", loss)
print("w.grad:", w.grad)
print("b.grad:", b.grad)

Prediction: tensor(6., grad_fn=<AddBackward0>)
Loss: tensor(16., grad_fn=<PowBackward0>)
w.grad: tensor(-24.)
b.grad: tensor(-8.)


## 18. A Complete Training Loop

The training loop is:

```text
Forward pass
    ↓
Loss
    ↓
backward()
    ↓
Gradients
    ↓
Parameter update
    ↓
Clear gradients
    ↓
Repeat
```

In [67]:
import torch

x = torch.tensor(3.0)
target = torch.tensor(10.0)

w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.01

for step in range(500):

    prediction = w * x + b
    loss = (prediction - target) ** 2

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    w.grad.zero_()
    b.grad.zero_()

    if step % 50 == 0:
        print(
            f"Step {step}: "
            f"w={w.item():.4f}, "
            f"b={b.item():.4f}, "
            f"loss={loss.item():.6f}"
        )

Step 0: w=2.2400, b=0.0800, loss=16.000000
Step 50: w=3.2000, b=0.4000, loss=0.000000
Step 100: w=3.2000, b=0.4000, loss=0.000000
Step 150: w=3.2000, b=0.4000, loss=0.000000
Step 200: w=3.2000, b=0.4000, loss=0.000000
Step 250: w=3.2000, b=0.4000, loss=0.000000
Step 300: w=3.2000, b=0.4000, loss=0.000000
Step 350: w=3.2000, b=0.4000, loss=0.000000
Step 400: w=3.2000, b=0.4000, loss=0.000000
Step 450: w=3.2000, b=0.4000, loss=0.000000


## 19. Learning Rate

A learning rate that is too large can cause the parameters to overshoot the minimum.

For this simple example, a learning rate of `0.1` produced an oscillation:

```text
(w=2.0, b=0.0)
        ↓
(w=4.4, b=0.8)
        ↓
(w=2.0, b=0.0)
        ↓
...
```

Reducing the learning rate to `0.01` produced smooth convergence.

The learning rate controls the size of each parameter update.

# Vectorized Training

The same neuron can process multiple training examples at once.

In [68]:
x = torch.tensor([1., 2., 3., 4., 5.])
target = torch.tensor([3., 5., 7., 9., 11.])

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

prediction = w * x + b
loss = ((prediction - target) ** 2).mean()

print("Prediction:", prediction)
print("Loss:", loss)

Prediction: tensor([0., 0., 0., 0., 0.], grad_fn=<AddBackward0>)
Loss: tensor(57., grad_fn=<MeanBackward0>)


The target relationship is:

$$
y=2x+1
$$

The model uses the same parameters `w` and `b` for all five examples.

In [69]:
loss.backward()

print("w.grad:", w.grad)
print("b.grad:", b.grad)

w.grad: tensor(-50.)
b.grad: tensor(-14.0000)


The gradients now combine information from **all training examples**.

This is the bridge from scalar examples to tensor-based neural-network training.

# PyTorch's Neural-Network Abstractions

We have now implemented the mechanics manually. PyTorch provides abstractions for the same ideas.

## 20. `nn.Linear`

Instead of manually creating:

```python
w = torch.tensor(..., requires_grad=True)
b = torch.tensor(..., requires_grad=True)
```

we can use:

```python
nn.Linear(1, 1)
```

which represents a linear transformation with a trainable weight and bias.

In [70]:
import torch.nn as nn

model = nn.Linear(1, 1)

print(model)
print(model.weight)
print(model.bias)

print(model.weight.requires_grad)
print(model.bias.requires_grad)

Linear(in_features=1, out_features=1, bias=True)
Parameter containing:
tensor([[0.1217]], requires_grad=True)
Parameter containing:
tensor([0.6426], requires_grad=True)
True
True


In [71]:
for name, parameter in model.named_parameters():
    print(name, parameter)

weight Parameter containing:
tensor([[0.1217]], requires_grad=True)
bias Parameter containing:
tensor([0.6426], requires_grad=True)


`nn.Linear` packages the parameters and forward computation into an `nn.Module`.

The underlying mathematics has not changed:

$$
y=xW+b
$$

In [72]:
x = torch.tensor([[1.],
                  [2.],
                  [3.],
                  [4.],
                  [5.]])

prediction = model(x)

print(prediction)
print("Input shape:", x.shape)
print("Output shape:", prediction.shape)

tensor([[0.7643],
        [0.8859],
        [1.0076],
        [1.1293],
        [1.2510]], grad_fn=<AddmmBackward0>)
Input shape: torch.Size([5, 1])
Output shape: torch.Size([5, 1])


## 21. Loss Function and Optimizer

PyTorch also provides standard abstractions for the loss and parameter update.

```python
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
```

The resulting training loop becomes:

```text
forward
   ↓
loss
   ↓
backward
   ↓
optimizer.step()
   ↓
optimizer.zero_grad()
   ↓
repeat
```

In [73]:
x = torch.tensor([[1.],
                  [2.],
                  [3.],
                  [4.],
                  [5.]])

target = torch.tensor([[3.],
                       [5.],
                       [7.],
                       [9.],
                       [11.]])

model = nn.Linear(1, 1)

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for step in range(1000):

    prediction = model(x)
    loss = loss_fn(prediction, target)

    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if step % 100 == 0:
        print(
            f"Step {step}: "
            f"loss={loss.item():.6f}"
        )

print("\nLearned parameters:")
print("weight:", model.weight.item())
print("bias:", model.bias.item())

Step 0: loss=96.322296
Step 100: loss=0.033380
Step 200: loss=0.016956
Step 300: loss=0.008613
Step 400: loss=0.004375
Step 500: loss=0.002222
Step 600: loss=0.001129
Step 700: loss=0.000573
Step 800: loss=0.000291
Step 900: loss=0.000148

Learned parameters:
weight: 2.005628824234009
bias: 0.9796784520149231


# Key Takeaways

- `requires_grad=True` enables gradient tracking.
- PyTorch builds a computational graph from tracked operations.
- `grad_fn` represents the backward operation associated with a tensor.
- `backward()` traverses the graph and computes gradients.
- Gradients accumulate by default.
- Leaf tensors store gradients in `.grad` by default.
- `retain_grad()` allows intermediate gradients to be retained.
- `torch.no_grad()` temporarily disables gradient tracking.
- `detach()` creates a tensor sharing storage but disconnected from Autograd.
- `clone().detach()` creates independent storage without gradient tracking.
- Gradient descent uses gradients to update parameters.
- Multiple examples can be processed using vectorized tensor operations.
- `nn.Linear` packages a trainable linear transformation.
- `MSELoss` provides a standard loss function.
- Optimizers such as SGD automate parameter updates.
- The underlying mathematics remains the same even when PyTorch automates the mechanics.

The core training cycle is:

```text
data
 ↓
forward pass
 ↓
loss
 ↓
backward()
 ↓
gradients
 ↓
parameter update
 ↓
repeat
```